# Visual search pilot

A small run to set up the main one (`run_model.ipynb`). Qwen3-VL-2B-Thinking looks at a display,
thinks, then answers yes or no. Its reaction time is the number of tokens it thinks for
(`observer.py`).

- **Displays:** 4 layouts per task and set size, each with and without the target: 64 displays
- **What to look at:** does every reply close its thinking and say yes or no? How many tokens does
  it think for, and does that grow with the number of items? How long does a batch take?

Results go to `../data/pilot.csv`. Colab: Runtime > Change runtime type > T4 GPU.

In [1]:
import os
import sys

inColab = "google.colab" in sys.modules

if inColab:
  from google.colab import drive
  drive.mount("/content/drive")
  os.chdir("/content/drive/MyDrive/Projects/vlm-psychophysics/chapters/visual_search/code")
  os.system("pip install -q -U transformers")

sys.path.insert(0, os.getcwd())
print("running in Colab:", inColab)

Mounted at /content/drive
running in Colab: True


In [2]:
import time
import torch
import numpy  as np
import pandas as pd
from transformers.utils import logging

import observer
import stimuli

logging.disable_progress_bar()
logging.set_verbosity_error()

device = "cuda" if torch.cuda.is_available() else "cpu"
obs    = observer.Observer(device=device)
print("device:", device, "| model:", observer.MODEL)

device: cpu | model: Qwen/Qwen3-VL-2B-Thinking


## The question

The chat as the model gets it. The image is one block of `<|image_pad|>` tokens; the model's reply
starts after `<think>`.

In [3]:
for task in stimuli.TASKS:
  print(obs.proc.apply_chat_template(obs.chat(task), tokenize=False, add_generation_prompt=True))

<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>Is there a red bar in the image? Answer yes or no.<|im_end|>
<|im_start|>assistant
<think>

<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>Is there a 2 in the image? Answer yes or no.<|im_end|>
<|im_start|>assistant
<think>



## The run

One batch = one task and set size: 4 layouts, each with and without the target (8 displays). If
the run stops, run the cell again: batches already in the file are skipped.

In [4]:
pilotFile = "../data/pilot.csv"
PAIRS     = 4

trials  = stimuli.make_trials(pairs=PAIRS, seed=1)
results = pd.read_csv(pilotFile).to_dict("records") if os.path.exists(pilotFile) else []
done    = set(r["index"] for r in results)

startTime = time.time()
for task in stimuli.TASKS:
  for n in stimuli.SET_SIZES:
    batch = [t for t in trials if t["task"] == task and t["set_size"] == n and t["index"] not in done]
    if not batch:
      continue
    t0 = time.time()
    answers = obs.ask(task, [stimuli.draw_trial(t) for t in batch], seed=batch[0]["index"])
    for trial, a in zip(batch, answers):
      said = {"yes": 1, "no": 0}.get(a["answer"])
      results.append({**trial, **a, "model": observer.MODEL,
                      "correct": int(said == trial["present"]) if said is not None else np.nan})
    pd.DataFrame(results).to_csv(pilotFile, index=False)
    print("%-8s %2d items: %3.0f s, longest thinking %d tokens"
          % (task, n, time.time() - t0, max(a["rt_tokens"] for a in answers)), flush=True)

print("done: %d displays in %.0f s" % (len(results), time.time() - startTime))

feature   3 items: 770 s, longest thinking 52 tokens
feature   6 items: 768 s, longest thinking 56 tokens
feature  12 items: 736 s, longest thinking 48 tokens
feature  18 items: 747 s, longest thinking 52 tokens
2v5       3 items: 1169 s, longest thinking 230 tokens
2v5       6 items: 977 s, longest thinking 152 tokens
2v5      12 items: 781 s, longest thinking 70 tokens
2v5      18 items: 1160 s, longest thinking 225 tokens
done: 64 displays in 7109 s


## What it did

Per task, set size and target present (1) or absent (0): proportion correct, thinking tokens
(median and mean), how many replies closed their thinking, and how many gave a yes or no.

In [5]:
df = pd.read_csv(pilotFile)
print(df.groupby(["task", "present", "set_size"]).agg(
    correct=("correct", "mean"), rt_median=("rt_tokens", "median"), rt_mean=("rt_tokens", "mean"),
    finished=("finished", "sum"), answered=("answer", "count"), displays=("index", "count"),
    reply_tokens=("reply_tokens", "mean")).round(2).to_string())
print()
print("answers:", df.answer.value_counts(dropna=False).to_dict())

                          correct  rt_median  rt_mean  finished  answered  displays  reply_tokens
task    present set_size                                                                         
2v5     0       3            1.00       64.0    98.50         4         4         4           2.0
                6            1.00       40.0    67.25         4         4         4           2.0
                12           1.00       50.0    48.00         4         4         4           2.0
                18           1.00       44.5    48.50         4         4         4           2.0
        1       3            1.00       48.5    52.50         4         4         4           2.0
                6            1.00       44.0    43.25         4         4         4           2.0
                12           0.25       56.0    58.00         4         4         4           2.0
                18           1.00      111.5   129.75         4         4         4           2.0
feature 0       3   

## Two replies in full

The thinking for a 2 among 17 5s, and for a red bar among 5 green bars.

In [6]:
for task, n in [("2v5", 18), ("feature", 6)]:
  r = df[(df.task == task) & (df.set_size == n) & (df.present == 1)].iloc[0]
  print("=" * 80)
  print("%s, %d items, target present: %d thinking tokens, answer %s" % (task, n, r.rt_tokens, r.answer))
  print("-" * 80)
  print(r.thinking)
  print("-" * 80)
  print("reply:", r.reply)

2v5, 18 items, target present: 225 thinking tokens, answer yes
--------------------------------------------------------------------------------
So, let's look at the image. The question is whether there's a 2. Scanning through each cell, the only number I see is 5, except for the middle row where there's a 2? Wait, no, wait. Let me check again. The image has multiple 5s. Wait, the bottom row has a 2? Let me see. The grid: first row: 5,5,5. Second row: 5,5,5,5. Third row: 5,5,5,5. Fourth row: 5,5,5,5. Fifth row: 5,5,5,5. Sixth row: 5,5,5,5. Wait, no, the fifth row (from top) has a 2? Wait, no, looking at the image, there's a "2" in the bottom row. Wait, the problem says "Is there a 2 in the image?" Let me confirm. The image shows numbers, and there's a 2. So the answer is yes.
--------------------------------------------------------------------------------
reply: yes
feature, 6 items, target present: 48 thinking tokens, answer yes
--------------------------------------------------------